<a href="https://colab.research.google.com/github/knc6/jarvis-tools-notebooks/blob/master/jarvis-tools-notebooks/Simple_Silicon_atomic_structure_and_analysis_example.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Description of Atoms module and analysis

# Table of contents
1. Creating Silicon Atoms class
2. Basic Atoms properties
3. Atoms class to_dict and from_dict methods
4. Making supercells
5. Spacegroup information and conventional structure
6. Making point-defect vacancy
7. Making surface structure
8. Making interfaces
9. Neighborhood analysis
10. X-ray diffractio (XRD) patterns
11. Obtaining machine-learning descriptors
12. Get an atomistic graph

In [ ]:
!pip install -q jarvis-tools spglib ase

### A crystal structure is defined by lattice-matrix, atomic coordinates and element types. As an example, let's create Siliocn crystal structure. We need to to import Atoms class from jarvis-tools to do that.

In [ ]:
from jarvis.core.atoms import Atoms

In [ ]:
#Silicon crystal structure example. These crystal-structure data are generally obtained from XRD-type experiments.
box = [[2.715, 2.715, 0], [0, 2.715, 2.715], [2.715, 0, 2.715]]
coords = [[0, 0, 0], [0.25, 0.25, 0.25]]
elements = ["Si", "Si"]
Si = Atoms(lattice_mat=box, coords=coords, elements=elements)

In [ ]:
from ase.visualize import view
view(Si.make_supercell([2,2,2]).ase_converter(),viewer='x3d')

In [ ]:
from ase.visualize import view
view(Si.get_conventional_atoms.make_supercell([2,2,2]).ase_converter(),viewer='x3d')

In [ ]:
# RDF
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
atoms=Si

dists=[]
for i in atoms.get_all_neighbors(r=10):
    for j in i:
      dists.append(j[2])
dists=np.array(dists)
hist, bins = np.histogram(dists, bins=np.arange(0.1, 10.2, 0.1))
number_density = atoms.num_atoms / atoms.volume
shell_vol = (
            4.0
            / 3.0
            * np.pi
            * (np.power(bins[1:], 3) - np.power(bins[:-1], 3))
        )

rdf = (
            hist / shell_vol / number_density / atoms.num_atoms
        )
plt.plot(bins[:-1], rdf)
plt.xlim([0,10])
plt.xlabel('Distance (Angstrom)')
plt.ylabel('RDF')
plt.show()

### Once the Atoms class is created, several properties can be calculated. Examples are given below:

In [ ]:
print ('volume',Si.volume)
print ('density in g/cm3', Si.density)
print ('composition as dictionary', Si.composition)
print ('Chemical formula', Si.composition.reduced_formula)
print ('Spacegroup info', Si.spacegroup())
print ('lattice-parameters', Si.lattice.abc, Si.lattice.angles)
print ('packing fraction',Si.packing_fraction)
print ('number of atoms',Si.num_atoms)
print ('Center of mass', Si.get_center_of_mass())
print ('Atomic number list', Si.atomic_numbers)

In [ ]:
# Print Atoms class in POSCAR format
print (Si)

### For databases purposes, it maybe important to store the Atoms class as a dictionary and then reloading the dictionary as Atoms object. An example is given below:

In [ ]:
Si.to_dict()

In [ ]:
d = Si.to_dict()
new_atoms = Atoms.from_dict(d)

Making a supercells: there are two methods

In [ ]:
supercell_1 = Si.make_supercell([2,2,2])
supercell_2 = Si.make_supercell_matrix([[2,0,0],[0,2,0],[0,0,2]])
supercell_1.density == supercell_2.density

### Many tensor-like material properties such as elastic constants require conventional cell representations, which can be obtained as follows:

In [ ]:
from jarvis.analysis.structure.spacegroup import Spacegroup3D

spg = Spacegroup3D(Si)
Si_cvn = spg.conventional_standard_structure
print ('Number of atoms in the conventional cell', Si_cvn.num_atoms)
print ('Wycoff multoplicity in the primitive-cell', spg._dataset["wyckoffs"])

In [ ]:
# Example to create Atoms object from POSCAR format file
from jarvis.io.vasp.inputs import Poscar
pos = """FCC Al
1.0
4.06741 0.0 0.0
0.0 4.06741 0.0
0.0 0.0 4.06741
Al
4
direct
0.0 0.0 0.0
0.0 0.5 0.5
0.5 0.0 0.5
0.5 0.5 0.0
"""
pos = Poscar.from_string(pos).atoms
print (pos)

In [ ]:
from ase.visualize import view
view(pos.ase_converter(),viewer='x3d')

### Realistic materials contain several types of point, line, surface and volume defects. Examples to create point-defects and surfaces are shown below:

In [ ]:
from jarvis.analysis.defects.vacancy import Vacancy

vacs = Vacancy(atoms=Si).generate_defects(enforce_c_size=10.0) #enforces cell-size to be close to 10 Angstroms
print (len(vacs), Atoms.from_dict(vacs[0].to_dict()["defect_structure"]).num_atoms)
# We find that there are only one unique point vacanc available based on Wycoff-position information

In [ ]:
from jarvis.analysis.defects.surface import wulff_normals, Surface

# Let's create (1,1,1) surface with three layers, and vacuum=18.0 Angstrom
# We center it around origin so that it looks good during visualization
surface_111 = Surface(atoms=Si, indices=[1, 1, 1], layers = 3, vacuum = 18 ).make_surface().center_around_origin()

In [ ]:
print (surface_111)

In [ ]:
from ase.visualize import view
view(surface_111.ase_converter(),viewer='x3d')

### For practical applications, materials are generally used as heterostructures. We can use the following as an example to make Si(111)-Si(111) interface

In [ ]:
from jarvis.analysis.interface.zur import ZSLGenerator, mismatch_strts, get_hetero,make_interface

film = Surface(atoms=Si, indices=[1, 1, 1], layers = 3, vacuum = 18 ).make_surface().center_around_origin()
substrate = Surface(atoms=Si, indices=[1, 1, 1], layers = 3, vacuum = 18 ).make_surface().center_around_origin()
info = make_interface(film=film, subs=substrate)['interface'].center(vacuum=18)

In [ ]:
info

### Atomic-neighborhood analysis

In [ ]:
from jarvis.analysis.structure.neighbors import NeighborsAnalysis

nb = NeighborsAnalysis(Si)

In [ ]:
bins_rdf, rdf, nbs = nb.get_rdf() #Global Radial distribution function
adfa, bins_a = nb.ang_dist_first() #Angular distribution function upto first neighbor
adfb, bins_b = nb.ang_dist_second() #Angular distribution function upto second neighbor
ddf, bins_d = nb.get_ddf() #Dihedral distribution function upto first neighbor

In [ ]:
import matplotlib
%matplotlib inline
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec

the_grid = GridSpec(2, 2)
plt.rcParams.update({'font.size': 24})
plt.figure(figsize=(16,14))

plt.subplot(the_grid[0, 0])
plt.title('(a) RDF')
plt.plot(bins_rdf, rdf)
plt.xlabel(r'Distance bins ($\AA$)')

plt.subplot(the_grid[0, 1])
plt.title('(b) ADF-a')
plt.plot(bins_a[:-1], adfa)
plt.xlabel(r'Angle bins ($^\circ$)')

plt.subplot(the_grid[1, 0])
plt.title('(c) ADF-b')
plt.plot(bins_b[:-1], adfb)
plt.xlabel(r'Angle bins ($^\circ$)')

plt.subplot(the_grid[1, 1])
plt.title('(d) DDF')
plt.plot(bins_d[:-1], ddf)
plt.xlabel(r'Angle bins ($^\circ$)')
plt.tight_layout()

### X-ray diffraction

In [ ]:
from jarvis.analysis.diffraction.xrd import XRD
theta,d_hkls,intens = XRD().simulate(atoms=Si)

In [ ]:
import matplotlib.pyplot as plt

# Assuming `theta` and `intens` are defined lists or arrays of the same length.
plt.bar(theta, intens)
plt.xlim(0, 90)
plt.xlabel(r'2$\theta$ (degrees)')
plt.ylabel('Intensity (a.u.)')
plt.show()


In [ ]:
from jarvis.core.spectrum import Spectrum
import numpy as np
def gaussian(theta2, alpha, fwhm):

    """
    Gaussian function for profiling peaks
    """

    tmp = ((alpha - theta2)/fwhm)**2
    return np.exp(-4*np.log(2)*tmp)

def get_profile(fwhm=0.1,min2theta=0,max2theta=180,res=0.02,two_thetas=[],intensities=[]):
    N = int((max2theta-min2theta)/res)
    px = np.linspace(min2theta, max2theta, N)
    py = np.zeros((N))
    for two_theta, intensity in zip(two_thetas, intensities):
        tmp = gaussian(two_theta, px, fwhm)
        py += intensity * tmp
    py /= np.max(py)
    return px,py

px,py=get_profile(two_thetas=theta,intensities=intens)

In [ ]:
plt.plot(px,py)

### Getting CFID descriptors: We can convert the chemo-physical information in the Atoms class to CFID features that can be used in Machine-learning

In [ ]:
from jarvis.ai.descriptors.cfid import CFID

cfid = CFID(Si)
feature_names = cfid.get_comp_descp(print_names=True)
feature_values = cfid.get_comp_descp().tolist()

In [ ]:
print (feature_names[0], feature_values[0], len(feature_names), len(feature_values) )

Get an atomistic graph

In [ ]:
!pip uninstall dgl -y
!pip install -q dgl -f https://data.dgl.ai/wheels/torch-2.1/repo.html


In [ ]:
from jarvis.core.graphs import Graph
g,lg = Graph.atom_dgl_multigraph(atoms=Si)

In [ ]:
g #graph

In [ ]:
lg #line graph

In [ ]:
g.ndata

Exercise: similar to Silicon, make surface, vacancy structures, RDF and XRD plots for Al, Al2O3. Find the atomic structures/POSCARs from JARVIS-DFT.